# Federated ADMET Demo — Walkthrough

This notebook walks through the three experiments end-to-end:

1. Centralised baseline (oracle: all data in one place)
2. Single-client baseline (1/N of the data, no federation)
3. Federated training across 3 simulated partner nodes
4. Membership inference attack against the federated model

Total runtime on CPU: ~10–20 minutes for `Caco2_Wang` (the smallest TDC ADMET task).

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
from src.data import load_admet_task, partition_data, smiles_list_to_graphs, ATOM_FEATURE_DIM, REGRESSION_TASKS

TASK = 'Caco2_Wang'
IS_REGRESSION = TASK in REGRESSION_TASKS
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Task: {TASK} | Regression: {IS_REGRESSION} | Device: {DEVICE}')

## 2. Load and inspect the data

In [ ]:
data = load_admet_task(TASK)
print(f'Train: {len(data.train)}  Valid: {len(data.valid)}  Test: {len(data.test)}')
data.train.head()

## 3. Partition the training data across simulated partners

In [ ]:
shards = partition_data(data.train, num_clients=3, strategy='random', seed=42)
for i, s in enumerate(shards):
    print(f'Client {i}: {len(s)} molecules')

## 4. Featurise to PyG graphs

In [ ]:
g = smiles_list_to_graphs(['CCO', 'c1ccccc1'], [1.0, 0.5])
print(g[0])
print(f'Atom feature dim: {ATOM_FEATURE_DIM}')

## 5. Run the experiments

From the repo root:

```bash
python scripts/run_baselines.py --task Caco2_Wang
python scripts/run_federated.py --task Caco2_Wang
python scripts/run_mia.py --task Caco2_Wang
```

Then load and compare results below.

In [ ]:
import json
from pathlib import Path

out = Path('../outputs')
if (out / f'baselines_{TASK}.json').exists():
    baselines = json.loads((out / f'baselines_{TASK}.json').read_text())
    federated = json.loads((out / f'federated_history_{TASK}.json').read_text())
    mia = json.loads((out / f'mia_result_{TASK}.json').read_text())
    
    metric = 'mae' if IS_REGRESSION else 'auc'
    print(f'\n=== {TASK} test {metric} ===')
    print(f'  Centralised:   {baselines["centralised"][metric]:.4f}')
    print(f'  Federated:     {federated["test_metrics"][metric]:.4f}')
    print(f'  Single client: {baselines["single_client"][metric]:.4f}')
    print(f'\n=== Membership inference attack ===')
    print(f'  Attack AUC:    {mia["attack_auc"]:.4f}  (0.5 = random guessing)')
else:
    print('Run the scripts above first.')